# TML — A100 Layout Generic Worker
Reusable for any year. VPS remains controller/source of truth; Colab computes layout, uploads each result immediately, and self-restarts after transient failures.


In [ ]:
YEAR=1904
WORKERS=4
DOWNLOADERS=8
VPS_HOST='vibrant-lovelace.82-165-11-122.plesk.page'
VPS_USER='andre'
VPS_PORT=2222
BASE=f'/home/andre/GallicaJobs/gallica-{YEAR}-all-tennis/GALlica_{YEAR}_ALL_TENNIS'
CLAIM=f'{BASE}/00_MANIFEST/colab_layout_active_claim.tsv'
STOP=f'{BASE}/00_MANIFEST/colab_layout_quality_complete_{YEAR}.flag'
GENERATION=f'colab_a100_layout_{YEAR}_v3_vps_model'
print('CONFIG',YEAR,'workers',WORKERS,'downloaders',DOWNLOADERS,'base',BASE,flush=True)


In [ ]:
import os,shutil,subprocess,sys
REPO='/content/Tennis-OCR-Pipeline'; VENV='/content/tml-layout-py312'; VENV_PY=f'{VENV}/bin/python'
PADDLE_URL='https://paddle-whl.cdn.bcebos.com/stable/cu126/paddlepaddle-gpu/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'
PADDLE_WHL='/content/paddlepaddle_gpu-3.2.0-cp312-cp312-linux_x86_64.whl'; EXPECTED=1890365820
print('SETUP 1/4 repo',flush=True)
if os.path.isdir(REPO): subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
else: subprocess.run(['git','clone','-q','https://github.com/Tennismylife/Tennis-OCR-Pipeline.git',REPO],check=True)
print('SETUP 2/4 Python 3.12 isolated env',flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','uv'],check=True)
UV=shutil.which('uv'); assert UV
subprocess.run([UV,'python','install','3.12'],check=True)
if not os.path.exists(VENV_PY): subprocess.run([UV,'venv','--seed','--python','3.12',VENV],check=True)
print('SETUP 3/4 Paddle GPU 3.2.0',flush=True)
need=subprocess.run([VENV_PY,'-c',"import paddle,sys; sys.exit(0 if paddle.__version__=='3.2.0' and paddle.device.is_compiled_with_cuda() else 1)"]).returncode!=0
if need:
 if not os.path.exists(PADDLE_WHL) or os.path.getsize(PADDLE_WHL)!=EXPECTED: subprocess.run(['curl','-L','--fail','--retry','5','-C','-','--progress-bar','-o',PADDLE_WHL,PADDLE_URL],check=True)
 subprocess.run([VENV_PY,'-m','pip','install','--progress-bar','on',PADDLE_WHL],check=True)
else: print('Paddle already ready; reinstall skipped',flush=True)
print('SETUP 4/4 exact working layout stack',flush=True)
subprocess.run([VENV_PY,'-m','pip','install','--progress-bar','on','paddlex==3.7.2','paddleocr==3.7.0','opencv-contrib-python==4.10.0.84','paramiko>=3.5,<4'],check=True)
subprocess.run([VENV_PY,'-c',"import paddle,paddlex,cv2; print('ENV_READY',paddle.__version__,paddlex.__version__,cv2.__version__,'CUDA',paddle.device.is_compiled_with_cuda())"],check=True)


In [ ]:
from google.colab import files
import os
uploaded=files.upload()
if not uploaded: raise RuntimeError('Upload the dedicated Colab SFTP private key')
name,data=next(iter(uploaded.items()))
if name.endswith('.pub'): raise RuntimeError('Upload the private key, not .pub')
KEY_FILE='/content/tml_colab_key'; open(KEY_FILE,'wb').write(data); os.chmod(KEY_FILE,0o600)
print('KEY_FILE_READY',name,flush=True)


In [ ]:
import subprocess
subprocess.run(['git','-C',REPO,'pull','--ff-only'],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','--short','HEAD'],text=True).strip(); print('CODE',commit,flush=True)
cmd=[VENV_PY,'-u',f'{REPO}/colab/layout_watch_filekey.py','--vps-key-file',KEY_FILE,'--vps-host',VPS_HOST,'--vps-user',VPS_USER,'--vps-port',str(VPS_PORT),'--claim',CLAIM,'--stop-flag',STOP,'--workers',str(WORKERS),'--downloaders',str(DOWNLOADERS),'--poll','10','--generation-label',GENERATION]
print('STARTING_GENERIC_LAYOUT_SUPERVISOR',YEAR,'workers',WORKERS,'claim',CLAIM,flush=True)
subprocess.run(cmd,check=True)
